# 2D FDTD — CuPy Deep Dive

**Authors:** Vraj Patel (241110080), Vardhman Dwivedi (241060033)  
**Course:** IDC 606 — Fast Computational Hydrodynamics, IIT Kanpur

This notebook focuses exclusively on:
1. **NumPy vectorized** CPU implementation (baseline)
2. **CuPy RawKernel** GPU implementation
3. **FLOP calculation** and throughput analysis
4. **Detailed operation-by-operation breakdown** — what happens on CPU vs GPU for every line

## Setup

In [ ]:
!pip install cupy-cuda12x -q 2>/dev/null || pip install cupy-cuda11x -q 2>/dev/null
!nvidia-smi

In [ ]:
import numpy as np
import cupy as cp
import time

print(f'CuPy {cp.__version__}')
print(f'Device: {cp.cuda.runtime.getDeviceProperties(0)["name"]}')

## Physics & Grid Parameters

In [ ]:
# ── Physical constants ──
c0   = 3.0e8
mu0  = 4.0e-7 * np.pi
eps0 = 1.0 / (mu0 * c0**2)

# ── Grid ──
Nx = 200
Ny = 200
dx = 1e-3
dy = 1e-3

# ── Time stepping (CFL) ──
courant = 0.99
dt = courant / (c0 * np.sqrt(1.0/dx**2 + 1.0/dy**2))
n_steps = 400

# ── Source ──
src_x, src_y = Nx // 2, Ny // 2
spread = 12.0 * dt
t0 = 3.0 * spread

# ── Coefficients ──
coeff_hx = np.float32(dt / (mu0 * dy))
coeff_hy = np.float32(dt / (mu0 * dx))
coeff_ez = np.float32(dt / eps0)

print(f'Grid       : {Nx} x {Ny} = {Nx*Ny:,} cells')
print(f'Steps      : {n_steps}')
print(f'Cell size  : {dx*1e3:.1f} mm')
print(f'dt         : {dt:.4e} s (Courant = {courant})')

## FLOP Calculation

Count every add, subtract, multiply, divide as 1 FLOP.

| Operation | Cells | FLOPs/cell | Formula | Total |
|-----------|-------|-----------|---------|-------|
| Hx update | Nx × (Ny-1) | 3 | sub, mul, sub | 119,400 |
| Hy update | (Nx-1) × Ny | 3 | sub, mul, add | 119,400 |
| Ez update | (Nx-1) × (Ny-1) | 7 | 2 sub, 2 div, 1 sub, 1 mul, 1 add | 277,207 |
| **Total/step** | | | | **515,807** |
| **400 steps** | | | | **206.4 MFLOP** |

In [ ]:
flops_hx = Nx * (Ny - 1) * 3
flops_hy = (Nx - 1) * Ny * 3
flops_ez = (Nx - 1) * (Ny - 1) * 7
flops_per_step = flops_hx + flops_hy + flops_ez
flops_total = flops_per_step * n_steps

print(f'Hx: {flops_hx:>10,} FLOPs/step  ({Nx}×{Ny-1} cells × 3)')
print(f'Hy: {flops_hy:>10,} FLOPs/step  ({Nx-1}×{Ny} cells × 3)')
print(f'Ez: {flops_ez:>10,} FLOPs/step  ({Nx-1}×{Ny-1} cells × 7)')
print(f'Total/step: {flops_per_step:>10,}')
print(f'Total:      {flops_total:>10,} = {flops_total/1e6:.2f} MFLOP = {flops_total/1e9:.4f} GFLOP')

---

## Version 1: NumPy Vectorized — Full End-to-End

Everything timed: array allocation, computation, energy calculation, snapshot saving.
This is the complete CPU simulation from start to finish — nothing excluded.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  NumPy FDTD — FULL END-TO-END (everything timed)
# ══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

print(f'NumPy FDTD: {Nx}x{Ny}, {n_steps} steps — full end-to-end ...')
t_np_start = time.time()

# ── Allocate arrays (timed) ──
Ez_np = np.zeros((Nx, Ny), dtype=np.float64)
Hx_np = np.zeros((Nx, Ny), dtype=np.float64)
Hy_np = np.zeros((Nx, Ny), dtype=np.float64)
total_energy_np = np.zeros(n_steps)
snapshots_np = {}

snapshot_interval = 40
snapshot_steps = set(range(0, n_steps, snapshot_interval))
snapshot_steps.add(n_steps - 1)

# ── Main time loop ──
for n in range(n_steps):
    t = n * dt

    # H update (vectorized)
    Hx_np[:, :-1] -= float(coeff_hx) * (Ez_np[:, 1:] - Ez_np[:, :-1])
    Hy_np[:-1, :] += float(coeff_hy) * (Ez_np[1:, :] - Ez_np[:-1, :])

    # E update (vectorized)
    Ez_np[1:, 1:] += float(coeff_ez) * (
        (Hy_np[1:, 1:] - Hy_np[:-1, 1:]) / dx
      - (Hx_np[1:, 1:] - Hx_np[1:, :-1]) / dy
    )

    # Source injection
    Ez_np[src_x, src_y] = np.exp(-((t - t0) / spread) ** 2)

    # PEC boundaries
    Ez_np[0, :] = 0; Ez_np[-1, :] = 0
    Ez_np[:, 0] = 0; Ez_np[:, -1] = 0

    # Energy calculation (every step — same work as GPU version)
    U_E = 0.5 * eps0 * np.sum(Ez_np**2) * dx * dy
    U_H = 0.5 * mu0 * np.sum(Hx_np**2 + Hy_np**2) * dx * dy
    total_energy_np[n] = U_E + U_H

    # Snapshot (same frequency as GPU version)
    if n in snapshot_steps:
        snapshots_np[n] = Ez_np.copy()
        elapsed = time.time() - t_np_start
        print(f'  step {n:4d}/{n_steps}  |  U = {total_energy_np[n]:.4e} J  |  {elapsed:.3f} s')

t_np = time.time() - t_np_start
gflops_np = flops_total / t_np / 1e9

print(f'\nNumPy END-TO-END: {t_np:.4f} s  ({t_np/n_steps*1000:.3f} ms/step)')
print(f'Throughput:       {gflops_np:.4f} GFLOP/s')
print(f'Snapshots saved:  {len(snapshots_np)}')
print(f'Energy computed:  every step ({n_steps} values)')

---

## Version 2: CuPy RawKernel — Full End-to-End

Everything timed from scratch: kernel compilation, GPU memory allocation, H→D transfer,
computation, energy calculation, D→H snapshot transfers, synchronization.
Nothing excluded — this is what you'd measure if you ran the GPU code cold.

### What happens where
- **[CPU]** runs on host CPU
- **[GPU]** runs on device GPU
- **[H→D]** host-to-device transfer (CPU RAM → GPU VRAM)
- **[D→H]** device-to-host transfer (GPU VRAM → CPU RAM)
- **[GPU alloc]** allocates GPU memory
- **[SYNC]** blocks CPU until GPU finishes

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CuPy FDTD — FULL END-TO-END (everything timed from scratch)
# ══════════════════════════════════════════════════════════════

print(f'CuPy FDTD: {Nx}x{Ny}, {n_steps} steps — full end-to-end ...')
t_cu_start = time.time()

# ── [CPU] Step 1: Define CUDA kernels (lazy — compiled on first launch) ──
cupy_update_H = cp.RawKernel(r'''
extern "C" __global__
void update_H(float* Ez, float* Hx, float* Hy,
              float coeff_hx, float coeff_hy, int Nx, int Ny) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i < Nx && j < Ny - 1) {
        int idx = i * Ny + j;
        Hx[idx] -= coeff_hx * (Ez[idx + 1] - Ez[idx]);
    }
    if (i < Nx - 1 && j < Ny) {
        int idx = i * Ny + j;
        Hy[idx] += coeff_hy * (Ez[idx + Ny] - Ez[idx]);
    }
}
''', 'update_H')

cupy_update_Ez = cp.RawKernel(r'''
extern "C" __global__
void update_Ez(float* Ez, const float* Hx, const float* Hy,
               float coeff_ez, float dx, float dy,
               int Nx, int Ny, int src_x, int src_y,
               const float* pulse_arr, int step) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= Nx || j >= Ny) return;
    int idx = i * Ny + j;
    if (i >= 1 && j >= 1) {
        float dHy = (Hy[idx] - Hy[idx - Ny]) / dx;
        float dHx = (Hx[idx] - Hx[idx - 1]) / dy;
        Ez[idx] += coeff_ez * (dHy - dHx);
    }
    if (i == src_x && j == src_y) Ez[idx] = pulse_arr[step];
    if (i == 0 || i == Nx-1 || j == 0 || j == Ny-1) Ez[idx] = 0.0f;
}
''', 'update_Ez')

cupy_energy = cp.RawKernel(r'''
extern "C" __global__
void compute_energy(const float* Ez, const float* Hx, const float* Hy,
                    float* out, float eps0, float mu0,
                    float dx, float dy, int Nx, int Ny) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= Nx || j >= Ny) return;
    int idx = i * Ny + j;
    float ez=Ez[idx], hx=Hx[idx], hy=Hy[idx];
    float e = (0.5f*eps0*ez*ez + 0.5f*mu0*(hx*hx+hy*hy)) * dx * dy;
    atomicAdd(out, e);
}
''', 'compute_energy')
t_compile = time.time() - t_cu_start
print(f'  Kernel definition:  {t_compile*1000:.1f} ms [CPU — lazy, not compiled yet]')

# ── [GPU alloc] Step 2: Allocate GPU memory ──
Ez_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
Hx_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
Hy_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
energy_cu = cp.zeros(1, dtype=cp.float32)
t_alloc = time.time() - t_cu_start
print(f'  GPU allocation:     {(t_alloc - t_compile)*1000:.1f} ms [GPU alloc: {3*Nx*Ny*4/1024:.0f} KB]')

# ── [CPU + H→D] Step 3: Compute pulse on CPU, upload to GPU ──
pulse_np = np.array([np.exp(-((n*dt - t0)/spread)**2)
                     for n in range(n_steps)], dtype=np.float32)
pulse_cu = cp.array(pulse_np, dtype=cp.float32)  # H→D: 1600 bytes
t_upload = time.time() - t_cu_start
print(f'  Pulse compute+H→D: {(t_upload - t_alloc)*1000:.1f} ms [CPU exp() + 1.6 KB upload]')

# ── [CPU] Step 4: Pre-cast scalar args ──
_chx  = np.float32(coeff_hx)
_chy  = np.float32(coeff_hy)
_cez  = np.float32(coeff_ez)
_dx   = np.float32(dx)
_dy   = np.float32(dy)
_eps0 = np.float32(eps0)
_mu0  = np.float32(mu0)
_Nx   = np.int32(Nx)
_Ny   = np.int32(Ny)
_sx   = np.int32(src_x)
_sy   = np.int32(src_y)

BLOCK = (8, 32)
GRID  = ((Nx + 8 - 1) // 8, (Ny + 32 - 1) // 32)

# ── Storage for results (on CPU) ──
snapshot_interval = 40
snapshot_steps = set(range(0, n_steps, snapshot_interval))
snapshot_steps.add(n_steps - 1)
total_energy_cu = np.zeros(n_steps)
snapshots_cu = {}

# ── [GPU] Step 5: Main simulation loop ──
print(f'  Starting simulation loop ({n_steps} steps) ...')
t_loop_start = time.time()

for n in range(n_steps):
    # [GPU kernel] H update
    cupy_update_H(GRID, BLOCK,
                  (Ez_cu, Hx_cu, Hy_cu, _chx, _chy, _Nx, _Ny))

    # [GPU kernel] Ez + source + PEC
    cupy_update_Ez(GRID, BLOCK,
                   (Ez_cu, Hx_cu, Hy_cu, _cez, _dx, _dy,
                    _Nx, _Ny, _sx, _sy, pulse_cu, np.int32(n)))

    # Energy + snapshots (same frequency as NumPy version)
    # Energy computed every step to match NumPy
    energy_cu[:] = 0
    cupy_energy(GRID, BLOCK,
                (Ez_cu, Hx_cu, Hy_cu, energy_cu,
                 _eps0, _mu0, _dx, _dy, _Nx, _Ny))
    cp.cuda.Device().synchronize()  # [SYNC]
    total_energy_cu[n] = float(energy_cu.get()[0])  # [D→H] 4 bytes

    if n in snapshot_steps:
        snapshots_cu[n] = cp.asnumpy(Ez_cu).copy()  # [D→H] 160 KB
        elapsed = time.time() - t_cu_start
        print(f'    step {n:4d}/{n_steps}  |  U = {total_energy_cu[n]:.4e} J  |  {elapsed:.3f} s')

cp.cuda.Device().synchronize()  # [SYNC] final
t_cu = time.time() - t_cu_start
t_loop = time.time() - t_loop_start

gflops_cu = flops_total / t_cu / 1e9

print(f'\nCuPy END-TO-END: {t_cu:.4f} s  ({t_cu/n_steps*1000:.3f} ms/step)')
print(f'  Breakdown:')
print(f'    Kernel definition:  {t_compile*1000:.1f} ms')
print(f'    GPU allocation:     {(t_alloc-t_compile)*1000:.1f} ms')
print(f'    Pulse + upload:     {(t_upload-t_alloc)*1000:.1f} ms')
print(f'    Simulation loop:    {t_loop*1000:.1f} ms')
print(f'Throughput:       {gflops_cu:.4f} GFLOP/s')
print(f'Snapshots saved:  {len(snapshots_cu)}')
print(f'Energy computed:  every step ({n_steps} values)')

---

## Verification

In [ ]:
# Compare final fields
Ez_cu_final = cp.asnumpy(Ez_cu)
diff = np.max(np.abs(Ez_cu_final.astype(np.float64) - Ez_np))
rel  = diff / max(np.max(np.abs(Ez_np)), 1e-30)

print(f'Max |CuPy - NumPy|: {diff:.2e}')
print(f'Relative diff:      {rel:.2e}', ' (MATCH)' if rel < 1e-2 else ' (MISMATCH!)')

# Compare energy arrays
energy_diff = np.max(np.abs(total_energy_cu - total_energy_np.astype(np.float32)))
print(f'Max energy diff:    {energy_diff:.2e}')

---

## Final End-to-End Comparison

Both versions do identical work: allocate arrays, run 400 steps, compute energy every step,
save 11 snapshots. The only difference is CPU vs GPU execution.

In [ ]:
speedup = t_np / t_cu

print(f"{'='*70}")
print(f"  END-TO-END COMPARISON  ({Nx}x{Ny} grid, {n_steps} steps)")
print(f"  Both versions: allocate + compute + energy every step + 11 snapshots")
print(f"{'='*70}")
print(f"")
print(f"  {'Version':<24} {'Time':>10} {'ms/step':>10} {'GFLOP/s':>10}")
print(f"  {'-'*24} {'-'*10} {'-'*10} {'-'*10}")

for name, t in [('NumPy (CPU, float64)', t_np), ('CuPy (GPU, float32)', t_cu)]:
    ms = t * 1000
    mss = t / n_steps * 1000
    gf = flops_total / t / 1e9
    ts = f'{ms:.1f} ms' if ms < 1000 else f'{t:.2f} s'
    print(f"  {name:<24} {ts:>10} {mss:>9.4f}  {gf:>9.4f}")

print(f"")
print(f"  Speedup: {speedup:.1f}x (CuPy end-to-end vs NumPy end-to-end)")
print(f"")
print(f"  What's included in CuPy time:")
print(f"    - Kernel definition (lazy RawKernel)")
print(f"    - GPU memory allocation (480 KB)")
print(f"    - Pulse array compute on CPU + H→D upload (1.6 KB)")
print(f"    - 400 × 3 kernel launches (H + Ez + energy)")
print(f"    - 400 × sync + D→H energy (4 bytes each)")
print(f"    - 11 × D→H Ez snapshot (160 KB each)")
print(f"")
print(f"  What's included in NumPy time:")
print(f"    - Array allocation (3 × 320 KB float64)")
print(f"    - 400 × vectorized H + Ez + source + PEC")
print(f"    - 400 × energy (np.sum over full arrays)")
print(f"    - 11 × snapshot copy (Ez.copy())")
print(f"")
print(f"  FLOP budget: {flops_total:,} = {flops_total/1e9:.4f} GFLOP")
print(f"  Verification: {rel:.2e} {'(MATCH)' if rel < 1e-2 else '(MISMATCH!)'}")
print(f"{'='*70}")

## Snapshots (CuPy)

In [ ]:
extent_mm = [0, Nx*dx*1e3, 0, Ny*dy*1e3]
keys = sorted(snapshots_cu.keys())
ncols = min(5, len(keys))
nrows = (len(keys) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4.4*ncols, 4*nrows))
axes = np.array(axes).flat if nrows * ncols > 1 else [axes]

for ax, step in zip(axes, keys):
    data = snapshots_cu[step]
    vmax = max(0.01, np.max(np.abs(data)))
    im = ax.imshow(data.T, origin='lower', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, extent=extent_mm, aspect='equal')
    ax.set_title(f'Step {step} (t={step*dt*1e9:.2f} ns)', fontsize=9)
    ax.set_xlabel('x [mm]', fontsize=8); ax.set_ylabel('y [mm]', fontsize=8)
    ax.plot(src_x*dx*1e3, src_y*dy*1e3, 'k+', ms=6, mew=1)
    plt.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle('CuPy FDTD — Ez Snapshots', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## Energy Conservation (both versions overlaid)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
time_ns = np.arange(n_steps) * dt * 1e9
ax.plot(time_ns, total_energy_np, 'b-', lw=1.5, label='NumPy (CPU, float64)')
ax.plot(time_ns, total_energy_cu, 'r--', lw=1.5, label='CuPy (GPU, float32)')
ax.axvline(x=(t0 + 3*spread)*1e9, color='gray', ls='--', lw=0.8, label='Source off')
ax.set_xlabel('Time [ns]'); ax.set_ylabel('Total EM Energy [J]')
ax.set_title('Energy Conservation — NumPy vs CuPy (both curves should overlap)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()